# 04 | Family-held-out split and preprocessing

**Goal:** test generalisation to unseen *plan families*. Grouping by `layout_id` (the first submission) keeps the 0° and 45° runs of a layout together, but the 7 sibling layouts of one footprint and density still sit on both sides of the split, which leaks information. The primary protocol therefore groups by `family_id`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'; RESULTS = ROOT / 'results'
import sys; sys.path.insert(0, str(ROOT / 'src'))
df = pd.read_csv(DATA / 'layout_level_dataset.csv')
target = 'target_U_canopy_mean_m_s'
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
def split_report(cv, col):
    rows = []
    for k, (tr, te) in enumerate(cv.split(df, groups=df[col]), 1):
        a, b = df.iloc[tr], df.iloc[te]
        rows.append({'fold': k, 'train_cases': len(tr), 'test_cases': len(te), 'test_families': ','.join(sorted(b.family_id.unique())),
                     'families_in_both': len(set(a.family_id) & set(b.family_id)), 'layouts_in_both': len(set(a.layout_id) & set(b.layout_id))})
    return pd.DataFrame(rows)
print('PRIMARY: GroupKFold(5) on family_id'); fam = split_report(GroupKFold(5), 'family_id'); print(fam.to_string(index=False))
assert (fam.families_in_both == 0).all()

PRIMARY: GroupKFold(5) on family_id
 fold  train_cases  test_cases       test_families  families_in_both  layouts_in_both
    1          168          56 A0625,A4444,S16,S64                 0                0
    2          182          42       A35,S1111,S53                 0                0
    3          182          42     A25,S0625,S4444                 0                0
    4          182          42         A16,A64,S35                 0                0
    5          182          42       A1111,A53,S25                 0                0


## The leakage the family split removes

Under the layout split, every held-out layout has sibling layouts (same footprint and density, different height pattern) in training:

In [2]:
lay = split_report(GroupKFold(5), 'layout_id')
print(lay[['fold','train_cases','test_cases','families_in_both','layouts_in_both']].to_string(index=False))
print('\nLayout split: held-out layouts whose family also appears in training =', int((lay.families_in_both > 0).sum()), 'of 5 folds')

 fold  train_cases  test_cases  families_in_both  layouts_in_both
    1          178          46                16                0
    2          178          46                16                0
    3          180          44                16                0
    4          180          44                16                0
    5          180          44                16                0

Layout split: held-out layouts whose family also appears in training = 5 of 5 folds


In [3]:
loo = split_report(LeaveOneGroupOut(), 'family_id'); assert (loo.families_in_both == 0).all()
print('Leave-one-family-out:', len(loo), 'folds, each tests', loo.test_cases.unique().tolist(), 'cases; overlap =', int(loo.families_in_both.sum()))

Leave-one-family-out: 16 folds, each tests [14] cases; overlap = 0


## Preprocessing boundary

Imputation, scaling (Ridge only) and one-hot encoding sit inside each sklearn `Pipeline`, fitted on training folds only. The mean baseline is recomputed per fold. `src/train_models.py` asserts that no family appears in both train and test for every fold.

**Three protocols are reported.** *Family 5-fold* is primary. *Leave-one-family-out* is the strictest check. *Layout 5-fold* is kept only as a labelled interpolation result. A held-out density at the edge of the range (for example A0625) forces extrapolation, which tree models cannot do (Roberts et al., 2017, discuss this trade-off of blocked validation).